# AgriPulse — Week 10 Streaming Simulation

**Notebook:** `07_streaming_simulation.ipynb`  
**Project:** AgriPulse — Agricultural Market Analytics  
**Team:** Team 12

## Objective

Simulate incoming agricultural market events using JSON files,
Databricks Auto Loader, and Structured Streaming.

**Flow:** JSON Events → Landing → Auto Loader → Structured Streaming → Bronze → Live Metric

This notebook uses Kafka only as design awareness; Kafka itself is not implemented.

In [0]:
from pyspark.sql import functions as F

SOURCE_PATH = "/Volumes/agripulse/default/agri_streaming_source/"
LANDING_PATH = "/Volumes/agripulse/default/agri_streaming_landing/"
CHECKPOINT_PATH = "/Volumes/agripulse/default/agri_streaming_checkpoint/"
BRONZE_TABLE = "workspace.default.bronze_agripulse_market_events_stream"

print("Source:", SOURCE_PATH)
print("Landing:", LANDING_PATH)
print("Checkpoint:", CHECKPOINT_PATH)
print("Bronze:", BRONZE_TABLE)

Source: /Volumes/agripulse/default/agri_streaming_source/
Landing: /Volumes/agripulse/default/agri_streaming_landing/
Checkpoint: /Volumes/agripulse/default/agri_streaming_checkpoint/
Bronze: workspace.default.bronze_agripulse_market_events_stream


## 1. Create the Week 10 folders

These folders are separate from the original AgriPulse raw-data files.

In [0]:
from pyspark.sql import functions as F

BASE_PATH = "/Volumes/agripulse/default/agri_raw_files/"

SOURCE_PATH = BASE_PATH + "streaming_source/"
LANDING_PATH = BASE_PATH + "streaming_landing/"
CHECKPOINT_PATH = BASE_PATH + "streaming_checkpoint/"

BRONZE_TABLE = "workspace.default.bronze_agripulse_market_events_stream"

print("Source:", SOURCE_PATH)
print("Landing:", LANDING_PATH)
print("Checkpoint:", CHECKPOINT_PATH)
print("Bronze:", BRONZE_TABLE)

Source: /Volumes/agripulse/default/agri_raw_files/streaming_source/
Landing: /Volumes/agripulse/default/agri_raw_files/streaming_landing/
Checkpoint: /Volumes/agripulse/default/agri_raw_files/streaming_checkpoint/
Bronze: workspace.default.bronze_agripulse_market_events_stream


## 2. Reset the simulation

Run this once when starting a fresh Week 10 demo. Do not run it between
incremental streaming tests because it removes the checkpoint.

In [0]:
RESET_SIMULATION = True

if RESET_SIMULATION:
    dbutils.fs.rm(SOURCE_PATH, True)
    dbutils.fs.rm(LANDING_PATH, True)
    dbutils.fs.rm(CHECKPOINT_PATH, True)

    dbutils.fs.mkdirs(SOURCE_PATH)
    dbutils.fs.mkdirs(LANDING_PATH)
    dbutils.fs.mkdirs(CHECKPOINT_PATH)

    spark.sql(f"DROP TABLE IF EXISTS {BRONZE_TABLE}")
    print("Simulation reset completed.")
else:
    print("Reset skipped.")

Simulation reset completed.


## 3. Create JSON event files

Each line is one JSON event representing an agricultural market price update.

In [0]:
events_01 = (
    '{"event_id":"EVT-0001","event_timestamp":"2026-07-03T10:15:00+05:30","event_type":"PRICE_UPDATE","commodity_id":101,"market_id":201,"modal_price":2650.50}\n'
    '{"event_id":"EVT-0002","event_timestamp":"2026-07-03T10:20:00+05:30","event_type":"PRICE_UPDATE","commodity_id":102,"market_id":202,"modal_price":1850.00}\n'
    '{"event_id":"EVT-0003","event_timestamp":"2026-07-03T10:25:00+05:30","event_type":"PRICE_UPDATE","commodity_id":103,"market_id":201,"modal_price":3200.75}\n'
    '{"event_id":"EVT-0004","event_timestamp":"2026-07-03T10:30:00+05:30","event_type":"PRICE_UPDATE","commodity_id":101,"market_id":203,"modal_price":2700.00}\n'
    '{"event_id":"EVT-0005","event_timestamp":"2026-07-03T10:35:00+05:30","event_type":"PRICE_UPDATE","commodity_id":104,"market_id":202,"modal_price":1450.25}'
)

events_02 = (
    '{"event_id":"EVT-0006","event_timestamp":"2026-07-03T10:40:00+05:30","event_type":"PRICE_UPDATE","commodity_id":102,"market_id":201,"modal_price":1900.00}\n'
    '{"event_id":"EVT-0007","event_timestamp":"2026-07-03T10:45:00+05:30","event_type":"PRICE_UPDATE","commodity_id":105,"market_id":203,"modal_price":4100.50}\n'
    '{"event_id":"EVT-0008","event_timestamp":"2026-07-03T10:50:00+05:30","event_type":"PRICE_UPDATE","commodity_id":103,"market_id":202,"modal_price":3150.00}\n'
    '{"event_id":"EVT-0009","event_timestamp":"2026-07-03T10:55:00+05:30","event_type":"PRICE_UPDATE","commodity_id":104,"market_id":201,"modal_price":1525.75}\n'
    '{"event_id":"EVT-0010","event_timestamp":"2026-07-03T11:00:00+05:30","event_type":"PRICE_UPDATE","commodity_id":101,"market_id":202,"modal_price":2725.00}'
)

events_03 = (
    '{"event_id":"EVT-0011","event_timestamp":"2026-07-03T11:05:00+05:30","event_type":"PRICE_UPDATE","commodity_id":105,"market_id":201,"modal_price":4050.25}\n'
    '{"event_id":"EVT-0012","event_timestamp":"2026-07-03T11:10:00+05:30","event_type":"PRICE_UPDATE","commodity_id":102,"market_id":203,"modal_price":1950.50}\n'
    '{"event_id":"EVT-0013","event_timestamp":"2026-07-03T11:15:00+05:30","event_type":"PRICE_UPDATE","commodity_id":103,"market_id":201,"modal_price":3225.00}\n'
    '{"event_id":"EVT-0014","event_timestamp":"2026-07-03T11:20:00+05:30","event_type":"PRICE_UPDATE","commodity_id":104,"market_id":203,"modal_price":1490.00}\n'
    '{"event_id":"EVT-0015","event_timestamp":"2026-07-03T11:25:00+05:30","event_type":"PRICE_UPDATE","commodity_id":101,"market_id":201,"modal_price":2685.75}'
)

dbutils.fs.put(SOURCE_PATH + "market_events_01.json", events_01, overwrite=True)
dbutils.fs.put(SOURCE_PATH + "market_events_02.json", events_02, overwrite=True)
dbutils.fs.put(SOURCE_PATH + "market_events_03.json", events_03, overwrite=True)

print("Created 3 JSON event files.")

Wrote 774 bytes.
Wrote 774 bytes.
Wrote 774 bytes.
Created 3 JSON event files.


## 4. Inspect the JSON files

In [0]:
display(
    spark.read
        .format("json")
        .load(SOURCE_PATH)
        .orderBy("event_id")
)

commodity_id,event_id,event_timestamp,event_type,market_id,modal_price
101,EVT-0001,2026-07-03T10:15:00+05:30,PRICE_UPDATE,201,2650.5
102,EVT-0002,2026-07-03T10:20:00+05:30,PRICE_UPDATE,202,1850.0
103,EVT-0003,2026-07-03T10:25:00+05:30,PRICE_UPDATE,201,3200.75
101,EVT-0004,2026-07-03T10:30:00+05:30,PRICE_UPDATE,203,2700.0
104,EVT-0005,2026-07-03T10:35:00+05:30,PRICE_UPDATE,202,1450.25
102,EVT-0006,2026-07-03T10:40:00+05:30,PRICE_UPDATE,201,1900.0
105,EVT-0007,2026-07-03T10:45:00+05:30,PRICE_UPDATE,203,4100.5
103,EVT-0008,2026-07-03T10:50:00+05:30,PRICE_UPDATE,202,3150.0
104,EVT-0009,2026-07-03T10:55:00+05:30,PRICE_UPDATE,201,1525.75
101,EVT-0010,2026-07-03T11:00:00+05:30,PRICE_UPDATE,202,2725.0


## 5. Validate the event schema and source data

In [0]:
input_df = spark.read.format("json").load(SOURCE_PATH)

input_df.printSchema()

display(
    input_df.select(
        "event_id",
        "event_timestamp",
        "event_type",
        "commodity_id",
        "market_id",
        "modal_price"
    ).orderBy("event_id")
)

display(
    input_df.agg(
        F.count("*").alias("physical_rows"),
        F.countDistinct("event_id").alias("distinct_event_ids"),
        F.sum(F.when(F.col("event_id").isNull(), 1).otherwise(0)).alias("null_event_ids"),
        F.sum(F.when(F.col("modal_price").isNull(), 1).otherwise(0)).alias("null_modal_prices")
    )
)

root
 |-- commodity_id: long (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_timestamp: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- market_id: long (nullable = true)
 |-- modal_price: double (nullable = true)



event_id,event_timestamp,event_type,commodity_id,market_id,modal_price
EVT-0001,2026-07-03T10:15:00+05:30,PRICE_UPDATE,101,201,2650.5
EVT-0002,2026-07-03T10:20:00+05:30,PRICE_UPDATE,102,202,1850.0
EVT-0003,2026-07-03T10:25:00+05:30,PRICE_UPDATE,103,201,3200.75
EVT-0004,2026-07-03T10:30:00+05:30,PRICE_UPDATE,101,203,2700.0
EVT-0005,2026-07-03T10:35:00+05:30,PRICE_UPDATE,104,202,1450.25
EVT-0006,2026-07-03T10:40:00+05:30,PRICE_UPDATE,102,201,1900.0
EVT-0007,2026-07-03T10:45:00+05:30,PRICE_UPDATE,105,203,4100.5
EVT-0008,2026-07-03T10:50:00+05:30,PRICE_UPDATE,103,202,3150.0
EVT-0009,2026-07-03T10:55:00+05:30,PRICE_UPDATE,104,201,1525.75
EVT-0010,2026-07-03T11:00:00+05:30,PRICE_UPDATE,101,202,2725.0


physical_rows,distinct_event_ids,null_event_ids,null_modal_prices
15,15,0,0


## 6. Create the Auto Loader streaming DataFrame

Auto Loader watches the landing folder for newly arriving JSON files.

In [0]:
event_schema = (
    "event_id STRING, "
    "event_timestamp TIMESTAMP, "
    "event_type STRING, "
    "commodity_id INT, "
    "market_id INT, "
    "modal_price DOUBLE"
)

events_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .schema(event_schema)
        .load(LANDING_PATH)
        .select(
            "event_id",
            "event_timestamp",
            "event_type",
            "commodity_id",
            "market_id",
            "modal_price",
            F.col("_metadata.file_path").alias("_source_file_path"),
            F.col("_metadata.file_name").alias("_source_file_name"),
            F.current_timestamp().alias("_ingested_at")
        )
)

print("Auto Loader streaming DataFrame is ready.")

Auto Loader streaming DataFrame is ready.


## 7. Define the Structured Streaming write

`availableNow=True` processes currently available new data and then stops.
The checkpoint remembers the files already processed.

In [0]:
def run_streaming_simulation():
    query = (
        events_stream.writeStream
            .format("delta")
            .outputMode("append")
            .option("checkpointLocation", CHECKPOINT_PATH)
            .trigger(availableNow=True)
            .toTable(BRONZE_TABLE)
    )
    query.awaitTermination()
    print("Streaming simulation completed.")

## 8. Batch 1 — simulate the first file arriving

In [0]:
dbutils.fs.cp(
    SOURCE_PATH + "market_events_01.json",
    LANDING_PATH + "market_events_01.json"
)

print("Batch 1 arrived.")

Batch 1 arrived.


In [0]:
run_streaming_simulation()

Streaming simulation completed.


## 9. Inspect Batch 1 in Bronze

In [0]:
display(
    spark.table(BRONZE_TABLE)
        .orderBy("event_id")
)

event_id,event_timestamp,event_type,commodity_id,market_id,modal_price,_source_file_path,_source_file_name,_ingested_at
EVT-0001,2026-07-03T04:45:00.000Z,PRICE_UPDATE,101,201,2650.5,/Volumes/agripulse/default/agri_raw_files/streaming_landing/market_events_01.json,market_events_01.json,2026-10-06T13:16:59.454Z
EVT-0002,2026-07-03T04:50:00.000Z,PRICE_UPDATE,102,202,1850.0,/Volumes/agripulse/default/agri_raw_files/streaming_landing/market_events_01.json,market_events_01.json,2026-10-06T13:16:59.454Z
EVT-0003,2026-07-03T04:55:00.000Z,PRICE_UPDATE,103,201,3200.75,/Volumes/agripulse/default/agri_raw_files/streaming_landing/market_events_01.json,market_events_01.json,2026-10-06T13:16:59.454Z
EVT-0004,2026-07-03T05:00:00.000Z,PRICE_UPDATE,101,203,2700.0,/Volumes/agripulse/default/agri_raw_files/streaming_landing/market_events_01.json,market_events_01.json,2026-10-06T13:16:59.454Z
EVT-0005,2026-07-03T05:05:00.000Z,PRICE_UPDATE,104,202,1450.25,/Volumes/agripulse/default/agri_raw_files/streaming_landing/market_events_01.json,market_events_01.json,2026-10-06T13:16:59.454Z


In [0]:
display(
    spark.table(BRONZE_TABLE)
        .groupBy("_source_file_name")
        .agg(
            F.count("*").alias("physical_rows"),
            F.countDistinct("event_id").alias("distinct_event_ids")
        )
        .orderBy("_source_file_name")
)

_source_file_name,physical_rows,distinct_event_ids
market_events_01.json,5,5


## 10. Live Metric — Total Streaming Events

This is the primary Week 10 live metric.

In [0]:
display(
    spark.table(BRONZE_TABLE)
        .agg(F.count("*").alias("total_streaming_events"))
)

total_streaming_events
5


## 11. Optional Live Metric — Average Modal Price

In [0]:
display(
    spark.table(BRONZE_TABLE)
        .agg(
            F.round(F.avg("modal_price"), 2).alias(
                "average_streaming_modal_price"
            )
        )
)

average_streaming_modal_price
2370.3


## 12. Batch 2 — simulate another file arriving

The checkpoint remembers Batch 1, so the next streaming run should process
only the newly discovered file.

In [0]:
dbutils.fs.cp(
    SOURCE_PATH + "market_events_02.json",
    LANDING_PATH + "market_events_02.json"
)

print("Batch 2 arrived.")

Batch 2 arrived.


In [0]:
run_streaming_simulation()

Streaming simulation completed.


## 13. Validate incremental processing

In [0]:
display(
    spark.table(BRONZE_TABLE)
        .groupBy("_source_file_name")
        .agg(
            F.count("*").alias("physical_rows"),
            F.countDistinct("event_id").alias("distinct_event_ids")
        )
        .orderBy("_source_file_name")
)

_source_file_name,physical_rows,distinct_event_ids
market_events_01.json,5,5
market_events_02.json,5,5


In [0]:
display(
    spark.table(BRONZE_TABLE)
        .agg(F.count("*").alias("total_streaming_events"))
)

total_streaming_events
10


## 14. No-new-file test

Run the stream again without adding a new file. The Bronze row count should
remain unchanged.

In [0]:
rows_before = spark.table(BRONZE_TABLE).count()

run_streaming_simulation()

rows_after = spark.table(BRONZE_TABLE).count()

print("Rows before:", rows_before)
print("Rows after :", rows_after)

if rows_before == rows_after:
    print("PASS: No new file produced no new rows.")
else:
    print("CHECK: Row count changed unexpectedly.")

Streaming simulation completed.
Rows before: 10
Rows after : 10
PASS: No new file produced no new rows.


## 15. Batch 3 — simulate a third file arriving

In [0]:
dbutils.fs.cp(
    SOURCE_PATH + "market_events_03.json",
    LANDING_PATH + "market_events_03.json"
)

print("Batch 3 arrived.")

Batch 3 arrived.


In [0]:
run_streaming_simulation()

Streaming simulation completed.


## 16. Final streaming output

In [0]:
display(
    spark.table(BRONZE_TABLE)
        .select(
            "event_id",
            "event_timestamp",
            "event_type",
            "commodity_id",
            "market_id",
            "modal_price",
            "_source_file_name",
            "_ingested_at"
        )
        .orderBy("event_id")
)

event_id,event_timestamp,event_type,commodity_id,market_id,modal_price,_source_file_name,_ingested_at
EVT-0001,2026-07-03T04:45:00.000Z,PRICE_UPDATE,101,201,2650.5,market_events_01.json,2026-10-06T13:16:59.454Z
EVT-0002,2026-07-03T04:50:00.000Z,PRICE_UPDATE,102,202,1850.0,market_events_01.json,2026-10-06T13:16:59.454Z
EVT-0003,2026-07-03T04:55:00.000Z,PRICE_UPDATE,103,201,3200.75,market_events_01.json,2026-10-06T13:16:59.454Z
EVT-0004,2026-07-03T05:00:00.000Z,PRICE_UPDATE,101,203,2700.0,market_events_01.json,2026-10-06T13:16:59.454Z
EVT-0005,2026-07-03T05:05:00.000Z,PRICE_UPDATE,104,202,1450.25,market_events_01.json,2026-10-06T13:16:59.454Z
EVT-0006,2026-07-03T05:10:00.000Z,PRICE_UPDATE,102,201,1900.0,market_events_02.json,2026-10-06T13:17:42.038Z
EVT-0007,2026-07-03T05:15:00.000Z,PRICE_UPDATE,105,203,4100.5,market_events_02.json,2026-10-06T13:17:42.038Z
EVT-0008,2026-07-03T05:20:00.000Z,PRICE_UPDATE,103,202,3150.0,market_events_02.json,2026-10-06T13:17:42.038Z
EVT-0009,2026-07-03T05:25:00.000Z,PRICE_UPDATE,104,201,1525.75,market_events_02.json,2026-10-06T13:17:42.038Z
EVT-0010,2026-07-03T05:30:00.000Z,PRICE_UPDATE,101,202,2725.0,market_events_02.json,2026-10-06T13:17:42.038Z


## 17. Final per-file reconciliation

In [0]:
display(
    spark.table(BRONZE_TABLE)
        .groupBy("_source_file_name")
        .agg(
            F.count("*").alias("physical_rows"),
            F.countDistinct("event_id").alias("distinct_event_ids"),
            F.min("event_timestamp").alias("first_event_timestamp"),
            F.max("event_timestamp").alias("last_event_timestamp")
        )
        .orderBy("_source_file_name")
)

_source_file_name,physical_rows,distinct_event_ids,first_event_timestamp,last_event_timestamp
market_events_01.json,5,5,2026-07-03T04:45:00.000Z,2026-07-03T05:05:00.000Z
market_events_02.json,5,5,2026-07-03T05:10:00.000Z,2026-07-03T05:30:00.000Z
market_events_03.json,5,5,2026-07-03T05:35:00.000Z,2026-07-03T05:55:00.000Z


## 18. Final live metrics

In [0]:
display(
    spark.table(BRONZE_TABLE)
        .agg(
            F.count("*").alias("total_streaming_events"),
            F.countDistinct("event_id").alias("distinct_event_ids"),
            F.round(F.avg("modal_price"), 2).alias("average_streaming_modal_price"),
            F.round(F.min("modal_price"), 2).alias("minimum_modal_price"),
            F.round(F.max("modal_price"), 2).alias("maximum_modal_price")
        )
)

total_streaming_events,distinct_event_ids,average_streaming_modal_price,minimum_modal_price,maximum_modal_price
15,15,2576.95,1450.25,4100.5


## 19. Event-type and commodity metrics

In [0]:
display(
    spark.table(BRONZE_TABLE)
        .groupBy("event_type")
        .count()
        .orderBy("event_type")
)

display(
    spark.table(BRONZE_TABLE)
        .groupBy("commodity_id")
        .agg(
            F.count("*").alias("event_count"),
            F.round(F.avg("modal_price"), 2).alias("average_modal_price")
        )
        .orderBy("commodity_id")
)

event_type,count
PRICE_UPDATE,15


commodity_id,event_count,average_modal_price
101,4,2690.31
102,3,1900.17
103,3,3191.92
104,3,1488.67
105,2,4075.38


## 20. Final data-quality checks

In [0]:
display(
    spark.table(BRONZE_TABLE)
        .agg(
            F.count("*").alias("total_rows"),
            F.countDistinct("event_id").alias("distinct_event_ids"),
            F.sum(F.when(F.col("event_id").isNull(), 1).otherwise(0)).alias("null_event_ids"),
            F.sum(F.when(F.col("modal_price").isNull(), 1).otherwise(0)).alias("null_modal_prices"),
            F.sum(F.when(F.col("modal_price") <= 0, 1).otherwise(0)).alias("non_positive_prices")
        )
)

total_rows,distinct_event_ids,null_event_ids,null_modal_prices,non_positive_prices
15,15,0,0,0


## 21. Optional checkpoint inspection

Some Databricks runtimes support `cloud_files_state`. If supported, the
following query shows Auto Loader file-processing state. If unsupported,
the checkpoint is still used by the streaming query.

In [0]:
try:
    display(
        spark.sql(
            f"SELECT * FROM cloud_files_state('{CHECKPOINT_PATH}')"
        )
    )
except Exception as e:
    print("cloud_files_state is not available in this runtime.")
    print("The configured checkpoint is still used by Auto Loader.")

path,size,create_time,discovery_time,commit_time,archive_time,source_id,processed_time,ingestion_state,archive_mode,move_location
dbstorage-prod-yjzfzx6qup/uc/c8f3f0e9-76ad-4325-8cee-c1bc39e88185/ca713e9e-b5b7-42be-8791-ef048110a090/__unitystorage/catalogs/1418fc6c-566d-4bda-b9e7-c410b59caeb4/volumes/f4d7ed45-5bc3-4529-9ef1-4b41c32198ab/streaming_landing/market_events_03.json,774,2026-10-06T13:18:12.000Z,2026-10-06T13:18:17.234Z,null,null,0,2026-10-06T13:18:21.501Z,INGESTED,null,null
dbstorage-prod-yjzfzx6qup/uc/c8f3f0e9-76ad-4325-8cee-c1bc39e88185/ca713e9e-b5b7-42be-8791-ef048110a090/__unitystorage/catalogs/1418fc6c-566d-4bda-b9e7-c410b59caeb4/volumes/f4d7ed45-5bc3-4529-9ef1-4b41c32198ab/streaming_landing/market_events_02.json,774,2026-10-06T13:17:35.000Z,2026-10-06T13:17:40.597Z,2026-10-06T13:18:22.001Z,null,0,2026-10-06T13:17:44.911Z,INGESTED,null,null
dbstorage-prod-yjzfzx6qup/uc/c8f3f0e9-76ad-4325-8cee-c1bc39e88185/ca713e9e-b5b7-42be-8791-ef048110a090/__unitystorage/catalogs/1418fc6c-566d-4bda-b9e7-c410b59caeb4/volumes/f4d7ed45-5bc3-4529-9ef1-4b41c32198ab/streaming_landing/market_events_01.json,774,2026-10-06T13:16:46.000Z,2026-10-06T13:16:58.100Z,2026-10-06T13:17:45.528Z,null,0,2026-10-06T13:17:05.654Z,INGESTED,null,null


## 22. Week 10 validation checklist

The Week 10 sprint card requires:

- JSON event schema is clear.
- Streaming notebook runs.
- Streaming output exists.
- At least one live metric is shown.
- Kafka is design-only.
- AI Transparency Note is included in the Week 10 log.

This notebook additionally demonstrates incremental file processing and
no-new-file behavior.

In [0]:
print("WEEK 10 STREAMING SIMULATION")
print("----------------------------")
print("Bronze table:", BRONZE_TABLE)
print("Total rows:", spark.table(BRONZE_TABLE).count())
print("Auto Loader: enabled")
print("Structured Streaming: enabled")
print("Checkpoint: configured")
print("Live metric: total_streaming_events")
print("Kafka: design-only")
print("Week 10 simulation completed.")

WEEK 10 STREAMING SIMULATION
----------------------------
Bronze table: workspace.default.bronze_agripulse_market_events_stream
Total rows: 15
Auto Loader: enabled
Structured Streaming: enabled
Checkpoint: configured
Live metric: total_streaming_events
Kafka: design-only
Week 10 simulation completed.


In [0]:
display(
    spark.read
    .format("json")
    .load(LANDING_PATH)
)

commodity_id,event_id,event_timestamp,event_type,market_id,modal_price
101,EVT-0001,2026-07-03T10:15:00+05:30,PRICE_UPDATE,201,2650.5
102,EVT-0002,2026-07-03T10:20:00+05:30,PRICE_UPDATE,202,1850.0
103,EVT-0003,2026-07-03T10:25:00+05:30,PRICE_UPDATE,201,3200.75
101,EVT-0004,2026-07-03T10:30:00+05:30,PRICE_UPDATE,203,2700.0
104,EVT-0005,2026-07-03T10:35:00+05:30,PRICE_UPDATE,202,1450.25
102,EVT-0006,2026-07-03T10:40:00+05:30,PRICE_UPDATE,201,1900.0
105,EVT-0007,2026-07-03T10:45:00+05:30,PRICE_UPDATE,203,4100.5
103,EVT-0008,2026-07-03T10:50:00+05:30,PRICE_UPDATE,202,3150.0
104,EVT-0009,2026-07-03T10:55:00+05:30,PRICE_UPDATE,201,1525.75
101,EVT-0010,2026-07-03T11:00:00+05:30,PRICE_UPDATE,202,2725.0


In [0]:
from pyspark.sql import functions as F

display(
    spark.table("workspace.default.bronze_agripulse_market_events_stream")
    .select(
        "event_id",
        "event_timestamp",
        "commodity_id",
        "market_id",
        "modal_price",
        "_source_file_name",
        "_ingested_at"
    )
    .orderBy("event_id")
)

event_id,event_timestamp,commodity_id,market_id,modal_price,_source_file_name,_ingested_at
EVT-0001,2026-07-03T04:45:00.000Z,101,201,2650.5,market_events_01.json,2026-10-06T13:16:59.454Z
EVT-0002,2026-07-03T04:50:00.000Z,102,202,1850.0,market_events_01.json,2026-10-06T13:16:59.454Z
EVT-0003,2026-07-03T04:55:00.000Z,103,201,3200.75,market_events_01.json,2026-10-06T13:16:59.454Z
EVT-0004,2026-07-03T05:00:00.000Z,101,203,2700.0,market_events_01.json,2026-10-06T13:16:59.454Z
EVT-0005,2026-07-03T05:05:00.000Z,104,202,1450.25,market_events_01.json,2026-10-06T13:16:59.454Z
EVT-0006,2026-07-03T05:10:00.000Z,102,201,1900.0,market_events_02.json,2026-10-06T13:17:42.038Z
EVT-0007,2026-07-03T05:15:00.000Z,105,203,4100.5,market_events_02.json,2026-10-06T13:17:42.038Z
EVT-0008,2026-07-03T05:20:00.000Z,103,202,3150.0,market_events_02.json,2026-10-06T13:17:42.038Z
EVT-0009,2026-07-03T05:25:00.000Z,104,201,1525.75,market_events_02.json,2026-10-06T13:17:42.038Z
EVT-0010,2026-07-03T05:30:00.000Z,101,202,2725.0,market_events_02.json,2026-10-06T13:17:42.038Z


In [0]:
display(
    spark.table("workspace.default.bronze_agripulse_market_events_stream")
    .agg(
        F.count("*").alias("total_streaming_events")
    )
)

total_streaming_events
15


In [0]:
display(dbutils.fs.ls(LANDING_PATH))

path,name,size,modificationTime
dbfs:/Volumes/agripulse/default/agri_raw_files/streaming_landing/market_events_01.json,market_events_01.json,774,1791292606000
dbfs:/Volumes/agripulse/default/agri_raw_files/streaming_landing/market_events_02.json,market_events_02.json,774,1791292655000
dbfs:/Volumes/agripulse/default/agri_raw_files/streaming_landing/market_events_03.json,market_events_03.json,774,1791292692000


## Evidence screenshots

Capture and save:

1. `screenshots/week10_streaming_input_files.png`
   - JSON event files.

2. `screenshots/week10_streaming_output.png`
   - Bronze streaming output.

3. `screenshots/week10_live_metric.png`
   - Total streaming events metric.

Recommended repository files:

```text
notebooks/07_streaming_simulation.ipynb
streaming/structured_streaming_design.md
streaming/kafka_event_schema.json
weekly_logs/week10_log.md
screenshots/week10_streaming_input_files.png
screenshots/week10_streaming_output.png
screenshots/week10_live_metric.png
```